In [1]:

import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from scipy.signal import butter, filtfilt, resample_poly

# ----------------------- 配置 -----------------------
ROOT = Path(r"D:\Project_Github\audio_click_mil")

AUDIO_DIR = ROOT / "data" / "original_audio"
BAG_CSV = ROOT / "processed_data" / "balanced_bag_labels.csv"

OUTPUT_DIR = ROOT / "processed_data" / "instances"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

EXCLUDE_FILES = {12}
TARGET_SR = 200000          # 目标采样率 200kHz
LOW_CUTOFF = 5000           # 带通低 cutoff
HIGH_CUTOFF = 200000        # 带通高 cutoff（接近 Nyquist）

# ----------------------- 工具函数 -----------------------
def parse_file_num(filename: str) -> int:
    return int(Path(filename).stem[-2:])


def bandpass_filter(signal: np.ndarray, sr: int, low=5000, high=200000, order=6):
    """5kHz - 200kHz 带通滤波"""
    nyquist = 0.5 * sr
    low_norm = low / nyquist
    high_norm = high / nyquist
    b, a = butter(order, [low_norm, high_norm], btype='band')
    return filtfilt(b, a, signal)


def resample_audio(signal: np.ndarray, orig_sr: int, target_sr: int):
    """降采样到目标采样率"""
    if orig_sr == target_sr:
        return signal
    return resample_poly(signal, target_sr, orig_sr)


# ----------------------- 主流程 -----------------------
def build_instances():
    print("读取 balanced_bag_labels.csv ...")
    bag_df = pd.read_csv(BAG_CSV)

    # 按 file_num 分组处理
    for file_num, file_group in tqdm(bag_df.groupby("file_num"), desc="Processing files"):
        if file_num in EXCLUDE_FILES:
            continue

        audio_filename = file_group["audio_path"].iloc[0].split("\\")[-1]   # 提取文件名
        audio_path = AUDIO_DIR / audio_filename

        if not audio_path.exists():
            print(f"警告: 音频文件不存在 {audio_path}")
            continue

        # 读取音频
        try:
            audio, orig_sr = sf.read(str(audio_path))
        except Exception as e:
            print(f"读取失败 {audio_filename}: {e}")
            continue

        # 转单通道
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1).astype(np.float32)

        # Step 1: 带通滤波 5kHz-200kHz
        try:
            audio = bandpass_filter(audio, orig_sr, LOW_CUTOFF, HIGH_CUTOFF)
        except Exception as e:
            print(f"滤波失败 {audio_filename}: {e}")
            continue

        # Step 2: 降采样到 200kHz
        audio = resample_audio(audio, orig_sr, TARGET_SR)
        sr = TARGET_SR

        # 按 bag 处理
        for _, row in file_group.iterrows():
            bag_idx = int(row["bag_idx"])
            bag_label = int(row["bag_label"])
            bag_start_sec = float(row["bag_start_sec"])

            # 计算该 bag 在音频中的起始采样点
            bag_start_sample = int(bag_start_sec * sr)

            # 提取 60 秒音频（不足则补零）
            bag_samples = int(60 * sr)                     # 60秒 * 200kHz = 12,000,000 samples
            segment = audio[bag_start_sample : bag_start_sample + bag_samples]

            if len(segment) < bag_samples:
                pad_len = bag_samples - len(segment)
                segment = np.pad(segment, (0, pad_len), mode='constant')

            # 重塑为 (60, 200000)
            segment = segment.reshape(60, TARGET_SR).astype(np.float32)

            # 保存 .npy
            npy_name = f"file_{file_num:02d}_bag_{bag_idx:03d}_label_{bag_label}.npy"
            npy_path = OUTPUT_DIR / npy_name

            np.save(npy_path, segment)

    print(f"\n所有 bag 的实例 .npy 文件生成完成！")
    print(f"保存目录: {OUTPUT_DIR}")
    print(f"每个 .npy 形状: (60, 200000)  →  60 个实例 × 200000 个采样点")


if __name__ == "__main__":
    build_instances()

读取 balanced_bag_labels.csv ...


Processing files: 100%|██████████| 34/34 [22:22<00:00, 39.47s/it]


所有 bag 的实例 .npy 文件生成完成！
保存目录: D:\Project_Github\audio_click_mil\processed_data\instances
每个 .npy 形状: (60, 200000)  →  60 个实例 × 200000 个采样点
